# Post-Quantum Cryptography Demo
This notebook demonstrates key generation, encryption, and decryption using `cryptography.fernet`, with a Flask web app for user interaction.


In [1]:
import os
print(os.getcwd())

C:\Users\www\pqc_project


In [2]:
from cryptography.fernet import Fernet

# Generate a symmetric key
key = Fernet.generate_key()
print("Generated Key:", key.hex())

Generated Key: 3737496e4e56476642576d3164664e57565253347549372d4c53437166724f687743376331784e44357a493d


In [3]:
# Create a Fernet cipher instance
cipher = Fernet(key)

# Encrypt a message
message = b"Secure PQC message"
ciphertext = cipher.encrypt(message)
print("Ciphertext:", ciphertext.hex())

Ciphertext: 674141414141426f45385f4d6158776d46414255692d49755f4838766f694d53647a51614865416b562d794a7536624f3147712d47744978633953725467674b6f6f4f675637464d595a4c3745455646635979735361717449505078614651583644504f39415a3753616c306e78493570736a6c52316f3d


In [4]:
# Decrypt the ciphertext
decrypted_message = cipher.decrypt(ciphertext)
print("Decrypted Message:", decrypted_message.decode())

# Verify
assert decrypted_message == message
print("Decryption successful!")

Decrypted Message: Secure PQC message
Decryption successful!


In [1]:
from threading import Thread
from flask import Flask, render_template, request, session, redirect, url_for
from cryptography.fernet import Fernet, InvalidToken
import os
from flask_talisman import Talisman

# Initialize Flask app with explicit template folder
app = Flask(__name__, template_folder=os.path.join(os.getcwd(), 'templates'))
app.secret_key = os.urandom(24)
Talisman(app)

@app.route('/')
def index():
    # Clear template cache
    app.jinja_env.cache = {}
    print("Rendering index.html from templates folder")  # Debug message
    return render_template('index.html')

@app.route('/generate_keys', methods=['POST'])
def generate_keys():
    key = Fernet.generate_key()
    session['key'] = key
    return redirect(url_for('index'))

@app.route('/encrypt', methods=['POST'])
def encrypt():
    if 'key' not in session:
        return "Error: Please generate a key first.", 400
    try:
        message = request.form['message'].encode()
        if not message:
            return "Error: Message cannot be empty.", 400
        cipher = Fernet(session['key'])
        ciphertext = cipher.encrypt(message)
        return f"Ciphertext: {ciphertext.hex()}"
    except Exception as e:
        return f"Error: Invalid input. {str(e)}", 400

@app.route('/decrypt', methods=['POST'])
def decrypt():
    if 'key' not in session:
        return "Error: Please generate a key first.", 400
    try:
        ciphertext = bytes.fromhex(request.form['ciphertext'])
        cipher = Fernet(session['key'])
        decrypted_message = cipher.decrypt(ciphertext)
        return f"Decrypted Message: {decrypted_message.decode()}"
    except (InvalidToken, ValueError) as e:
        return f"Error: Invalid ciphertext or key. {str(e)}", 400

def run_app():
    app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

# Start Flask in a separate thread
thread = Thread(target=run_app)
thread.start()

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.100.177:5000
Press CTRL+C to quit


## Observations
- Used `cryptography.fernet` due to Windows compatibility issues with `liboqs-python`.
- Successfully implemented key generation, encryption, and decryption.
- Flask app provides a user-friendly UI with error handling and secure headers via Flask-Talisman.